# Notebook 02  Autoencoder Pre-training
**Phase 1:** Train CNN Encoder + Decoder on Normal clips only (MSE reconstruction loss).

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.config import (
    MODELS_DIR, PLOTS_DIR, OUTPUTS_DIR, SEED, CLIP_LEN, CLASS_TO_IDX,
)
from src.model import Encoder, Decoder, Autoencoder
from src.train import pretrain_autoencoder, plot_loss
from src.evaluate import (
    compute_reconstruction_errors, compute_threshold,
    plot_reconstruction_samples, plot_mse_distribution
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR = OUTPUTS_DIR / "clips_packed"
print(f"PyTorch: {torch.__version__}")
print(f"Device : {DEVICE}")


PyTorch: 2.13.0+cu130
Device : cuda


## 1. Load Data

In [2]:
meta_file = CLIPS_DIR / "meta.json"
with open(meta_file) as f:
    meta = json.load(f)

normal_npy     = meta["normal_pretrain"]["npy_paths"]
val_npy        = meta["val"]["npy_paths"]
y_val          = np.array(meta["val"]["y"])

val_normal_npy = [p for p, l in zip(val_npy, y_val) if l == CLASS_TO_IDX["Normal"]]
val_anom_npy   = [p for p, l in zip(val_npy, y_val) if l != CLASS_TO_IDX["Normal"]]

print(f"Normal pretrain clips : {len(normal_npy):,}  (packed .npy)")
print(f"Val Normal clips      : {len(val_normal_npy):,}  (packed .npy)")
print(f"Val Anomaly clips     : {len(val_anom_npy):,}  (packed .npy)")


Normal pretrain clips : 46,929  (packed .npy)
Val Normal clips      : 11,947  (packed .npy)
Val Anomaly clips     : 1,326  (packed .npy)


## 2. Build Autoencoder

In [3]:
encoder     = Encoder()
decoder     = Decoder()
autoencoder = Autoencoder(encoder, decoder)

total = sum(p.numel() for p in autoencoder.parameters())
print(f"Total parameters: {total:,}")
print(autoencoder)

Total parameters: 4,462,339
Autoencoder(
  (encoder): Encoder(
    (cnn): Sequential(
      (0): Conv2d(6, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (fc): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=8192, out_features=256, bias=True)
      (2): ReLU()
    )
  )
  (decoder): Decoder(
    (fc): Sequential(
      (0): Linear(in_features=256, out_features=8192, bias=True)
      (1): ReLU()
    )
    (deconv): Sequential(
      (0): ConvTranspose2d(128, 64, kernel_size

## 3. Pre-train

In [4]:
history = pretrain_autoencoder(
    autoencoder, normal_npy,
    val_split=0.15, device=DEVICE, use_packed=True,
)


Epoch 01/30  loss=0.03182  val_loss=0.02075
Epoch 05/30  loss=0.01083  val_loss=0.01535
Epoch 10/30  loss=0.00864  val_loss=0.01384
Epoch 15/30  loss=0.00763  val_loss=0.01324
Epoch 20/30  loss=0.00699  val_loss=0.01279
Epoch 25/30  loss=0.00654  val_loss=0.01258
Epoch 30/30  loss=0.00620  val_loss=0.01246


## 4. Loss Curves

In [5]:
plot_loss(history, title="Autoencoder Pre-training Loss",
          save_path=PLOTS_DIR / "pretrain_loss.png")
plt.show()

[OK] Plot saved: /home/mjl/softwarica/ANN/outputs/plots/pretrain_loss.png


## 5. Visual Quality Check

In [6]:
# Reload best weights
autoencoder.load_state_dict(
    torch.load(MODELS_DIR / "autoencoder_best.pth", map_location=DEVICE))
autoencoder = autoencoder.to(DEVICE)

# Load small packed sample for visualisation (no PNG loading needed)
def _load_packed_as_6ch(npy_path):
    rgb = np.load(npy_path).astype(np.float32) / 255.0  # (T, H, W, 3)
    diffs = np.diff(rgb, axis=0)
    diffs = np.concatenate([np.zeros_like(rgb[:1]), diffs], axis=0)
    return np.concatenate([rgb, diffs], axis=-1)  # (T, H, W, 6)

sample_normal = np.stack([_load_packed_as_6ch(p) for p in val_normal_npy[:8]])
sample_anom   = np.stack([_load_packed_as_6ch(p) for p in val_anom_npy[:8]])

plot_reconstruction_samples(autoencoder, sample_normal, n=4,
    save_path=PLOTS_DIR / "reconstructions_normal.png", device=DEVICE)
plt.show()

plot_reconstruction_samples(autoencoder, sample_anom, n=4,
    save_path=PLOTS_DIR / "reconstructions_anomaly.png", device=DEVICE)
plt.show()


[OK] Reconstructions saved: /home/mjl/softwarica/ANN/outputs/plots/reconstructions_normal.png
[OK] Reconstructions saved: /home/mjl/softwarica/ANN/outputs/plots/reconstructions_anomaly.png


## 6. MSE Distribution: Normal vs Anomalous

In [7]:
plot_mse_distribution(autoencoder, val_normal_npy, val_anom_npy,
    save_path=PLOTS_DIR / "mse_distribution.png", device=DEVICE)
plt.show()


[OK] MSE distribution saved: /home/mjl/softwarica/ANN/outputs/plots/mse_distribution.png


## 7. Compute and Save Thresholds

In [8]:
normal_errors = compute_reconstruction_errors(autoencoder, val_normal_npy, device=DEVICE)
thr_2std = compute_threshold(normal_errors, 2.0)
thr_3std = compute_threshold(normal_errors, 3.0)

print(f"Normal MSE  mean={normal_errors.mean():.5f}  std={normal_errors.std():.5f}")
print(f"Threshold (mean + 2*std) = {thr_2std:.5f}")
print(f"Threshold (mean + 3*std) = {thr_3std:.5f}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "thresholds.json", "w") as f:
    json.dump({"mean_2std": thr_2std, "mean_3std": thr_3std}, f, indent=2)
print("Thresholds saved.")
print("\n[DONE] Encoder saved: encoder_pretrained.pth  Decoder: decoder_pretrained.pth")


Normal MSE  mean=0.01382  std=0.00717
Threshold (mean + 2*std) = 0.02816
Threshold (mean + 3*std) = 0.03533


NameError: name 'RESULTS_DIR' is not defined